# reduce-gather-sum — ex2: global median across ranks via all_gather + manual sort

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-gather-sum`. Running the final beacon cell reports progress against the `Distributed: reduce.gather + sum` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce.gather + sum` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-gather-sum`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-gather-sum"
DD_SUBTOPIC = "Distributed: reduce.gather + sum"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `all_gather` + manual aggregation — quick refresher

`dist.all_gather(gather_list, tensor)` collects each rank's `tensor` into a length-`world_size` list of tensors on EVERY rank. Unlike `gather` (which populates only the dst rank), `all_gather` fans the result out — every rank ends with the same `gather_list`.

**Why all_gather instead of all_reduce.** `all_reduce` collapses to a single aggregate (sum, max, min, product). For aggregations that AREN'T associative-binary — median, percentile, sorted top-k — there's no `ReduceOp` you can pass. You need the per-rank values, on every rank, then compute the aggregation locally.

**Memory trade-off.** `all_gather` holds N tensors on every rank (N×memory). `all_reduce` holds 1 tensor on every rank. For scalar metrics across modest world sizes, the cost is negligible.

**Same gather_list pre-allocation pattern as `gather`.** Callers pre-build the `[t.zeros_like(...) for _ in range(world_size)]` list and pass it as the destination. The collective fills the slots.

### Exercise 2 — global median across ranks via all_gather + manual sort

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.all_gather` followed by a local sort + median to compute a global median across ranks — an aggregation `ReduceOp` cannot express.
> Keywords: all_gather, median, manual-aggregation, non-associative
> ```

**KCs targeted:** `all-gather-pre-allocates-gather-list`, `median-from-gathered-tensors`

Implement `ex2_all_gather_median(rank, world_size, dist_module, local_value)`. Compute the global MEDIAN of `world_size` rank-local scalars on every rank.

Steps:
1. Wrap the local value: `tensor = t.tensor([local_value], dtype=t.float32)`.
2. Pre-allocate the gather list: `gather_list = [t.zeros(1, dtype=t.float32) for _ in range(world_size)]`. Build this on EVERY rank (not just rank 0) — `all_gather` populates every rank's list.
3. `dist_module.all_gather(gather_list, tensor)`. After this, every rank's `gather_list[r]` holds rank r's value.
4. Stack into one tensor: `gathered = t.cat(gather_list)` (shape `(world_size,)`).
5. Sort and take the median index — for even `world_size`, average the two middle values:
   ```python
   sorted_vals = t.sort(gathered).values
   mid = world_size // 2
   if world_size % 2 == 1:
       median = sorted_vals[mid].item()
   else:
       median = ((sorted_vals[mid - 1] + sorted_vals[mid]) / 2).item()
   ```
6. Return `median` — a Python float, identical on every rank.

**Why `all_gather`, not `reduce`.** `ReduceOp` only supports associative binary ops (sum, max, min, product). Median requires the FULL distribution to compute — every rank needs the whole set, hence `all_gather`.

In [ ]:
def ex2_all_gather_median(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    gather_list = [t.zeros(1, dtype=t.float32) for _ in range(world_size)]
    dist_module.all_gather(gather_list, tensor)
    gathered = t.cat(gather_list)
    sorted_vals = t.sort(gathered).values
    mid = world_size // 2
    if world_size % 2 == 1:
        return sorted_vals[mid].item()
    return ((sorted_vals[mid - 1] + sorted_vals[mid]) / 2).item()


<details><summary>Solution</summary>

```python
def ex2_all_gather_median(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    gather_list = [t.zeros(1, dtype=t.float32) for _ in range(world_size)]
    dist_module.all_gather(gather_list, tensor)
    gathered = t.cat(gather_list)
    sorted_vals = t.sort(gathered).values
    mid = world_size // 2
    if world_size % 2 == 1:
        return sorted_vals[mid].item()
    return ((sorted_vals[mid - 1] + sorted_vals[mid]) / 2).item()
```

**The list-building step is the easiest place to introduce a bug.** `[t.zeros(1)] * world_size` creates `world_size` references to the SAME tensor — all `world_size` 'slots' alias and the gather silently overwrites itself. Always use a list comprehension to allocate distinct tensors.

**Median vs `kthvalue`.** `t.median` on an even-length tensor returns the LOWER of the two middle values, not their average. Above we compute the average explicitly so the answer matches numpy/scipy conventions. If you actually want torch's lower-mid behavior, `t.median(gathered).values` works for both odd and even.

**`gather` (single dst) vs `all_gather` (everyone).** Use `gather` when only rank 0 needs the result (then broadcast). Use `all_gather` when every rank needs to act on the per-rank values. For a metric you only log on rank 0, gather is cheaper.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()